In [5]:
# from google.colab import drive
# drive.mount('/content/drive')

In [6]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = 'seaborn'

from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline


In [7]:
df = pd.read_csv('50_Startups.csv')
df.head()

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   R&D Spend        50 non-null     float64
 1   Administration   50 non-null     float64
 2   Marketing Spend  50 non-null     float64
 3   State            50 non-null     object 
 4   Profit           50 non-null     float64
dtypes: float64(4), object(1)
memory usage: 2.1+ KB


In [9]:
# Plotly Pairplot
fig = px.scatter_matrix(df, dimensions=['R&D Spend', 'Administration', 'Marketing Spend', 'Profit'], width=1000, height=1000)
fig.show()

In [12]:
# Correlation Matrix
'''
 Try Different colors for heatmap
 ********************************
 ['aggrnyl', 'agsunset', 'algae', 'amp', 'armyrose', 'balance','blackbody', 'bluered', 'blues', 'blugrn', 'bluyl', 'brbg',
    'brwnyl', 'bugn', 'bupu', 'burg', 'burgyl', 'cividis', 'curl', 'darkmint', 'deep', 'delta', 'dense', 'earth', 'edge', 'electric',
    'emrld', 'fall', 'geyser', 'gnbu', 'gray', 'greens', 'greys', 'haline', 'hot', 'hsv', 'ice', 'icefire', 'inferno', 'jet',
    'magenta', 'magma', 'matter', 'mint', 'mrybm', 'mygbm', 'oranges', 'orrd', 'oryel', 'oxy', 'peach', 'phase', 'picnic', 'pinkyl',
    'piyg', 'plasma', 'plotly3', 'portland', 'prgn', 'pubu', 'pubugn', 'puor', 'purd', 'purp', 'purples', 'purpor', 'rainbow', 'rdbu',
    'rdgy', 'rdpu', 'rdylbu', 'rdylgn', 'redor', 'reds', 'solar', 'spectral', 'speed', 'sunset', 'sunsetdark', 'teal', 'tealgrn',
    'tealrose', 'tempo', 'temps', 'thermal', 'tropic', 'turbid', 'turbo', 'twilight', 'viridis', 'ylgn', 'ylgnbu', 'ylorbr','ylorrd'].
'''

fig = px.imshow(df.corr(numeric_only=True), width=700, height=700, color_continuous_scale='blugrn', title='Correlation Matrix')

# add annotations with the correlation values
for i in range(len(df.corr(numeric_only=True))):
    for j in range(len(df.corr(numeric_only=True))):
        fig.add_annotation(x=i, y=j, text="{:.2f}".format(df.corr(numeric_only=True).iloc[i,j]), showarrow=False, font_size=14)

        
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [13]:
df.corr(numeric_only=True)['Profit'].sort_values(ascending=False)

Profit             1.000000
R&D Spend          0.972900
Marketing Spend    0.747766
Administration     0.200717
Name: Profit, dtype: float64

In [14]:
X = df[['R&D Spend']]
y = df['Profit']

In [15]:
# Splitting the dataset into the Training set and Test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

In [16]:
# Feature Scaling
sc = StandardScaler()

X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [17]:
# Try Linear Regression
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train, y_train)

# Evaluate the model
print('LR Training Score: ', lr.score(X_train, y_train))
print('LR Test Score: ', lr.score(X_test, y_test))


LR Training Score:  0.9449589778363044
LR Test Score:  0.9464587607787219


In [18]:
lr.intercept_, lr.coef_

(109446.44725, array([39336.27887824]))

In [19]:
# Visualising the Linear Regression results
x = pd.DataFrame(np.arange(df['R&D Spend'].min(), df['R&D Spend'].max(), 1000), columns=['R&D Spend'])
x_sc= sc.transform(x)
y = lr.predict(x_sc)

fig = px.scatter(df , x='R&D Spend', y='Profit', title='Linear Regression', width=700, height=500)
fig.add_trace(go.Scatter(x=x['R&D Spend'], y=y, mode='lines'))
fig.update_layout(showlegend=False)
fig.show()

In [20]:
# SvR
svr = SVR(kernel='linear')
svr.fit(X_train, y_train)

# Evaluate the model
print('SVR Training Score: ', svr.score(X_train, y_train))
print('SVR Test Score: ', svr.score(X_test, y_test))


SVR Training Score:  0.0002443108207712541
SVR Test Score:  -0.15811411198826186


In [21]:
svr.intercept_, svr.coef_

(array([107978.75886052]), array([[32.47912661]]))

In [25]:
# Tuning the model
from sklearn.model_selection import GridSearchCV

parameters = {'C': [100, 1000, 10000], 'epsilon': [0.3, 0.1, 0.01, 0.001]}

grid_search = GridSearchCV(estimator = SVR(kernel='linear'), param_grid = parameters, scoring = 'r2', cv = 3, n_jobs = -1)
grid_search = grid_search.fit(X_train, y_train)

best_parameters = grid_search.best_params_
best_accuracy = grid_search.best_score_

print('Best Parameters: ', best_parameters)
print('Best Accuracy: ', best_accuracy)

Best Parameters:  {'C': 10000, 'epsilon': 0.001}
Best Accuracy:  0.9216816335941177


In [26]:
svr = SVR(**best_parameters, kernel='linear')
svr.fit(X_train, y_train)

# Evaluate the model
print('SVR Training Score: ', svr.score(X_train, y_train))
print('SVR Test Score: ', svr.score(X_test, y_test))

SVR Training Score:  0.9400035533371356
SVR Test Score:  0.9375147622038652


In [27]:
svr.intercept_, svr.coef_

(array([109433.92195042]), array([[36487.73423688]]))

In [28]:
# Visualising the SVR results
x = pd.DataFrame(np.arange(df['R&D Spend'].min(), df['R&D Spend'].max(), 1000), columns=['R&D Spend'])
x_sc= sc.transform(x)
y = svr.predict(x_sc)

fig = px.scatter(df , x='R&D Spend', y='Profit', title='Support Vector Regressor', width=700, height=500)
fig.add_trace(go.Scatter(x=x['R&D Spend'], y=y, mode='lines'))
fig.update_layout(showlegend=False)
fig.show()

In [30]:
x = pd.DataFrame(columns=['R&D Spend'], data=[[165349.20]])
x_sc = sc.transform(x)
print('Predicted Profit from SVR: ', svr.predict(x_sc))
print('Predicted Profit from LR: ', lr.predict(x_sc))


Predicted Profit from SVR:  [183441.27407631]
Predicted Profit from LR:  [189231.44632296]


# Non Linear Data

In [32]:
np.random.seed(42)
m = 100
X = 6 * np.random.rand(m, 1) - 3
y = 0.5 * X ** 2 + X + 2 + np.random.randn(m, 1)


px.scatter(x=X.flatten(), y=y.flatten(), title='Quadratic-looking data', labels={'x': 'X', 'y': 'y'}, width=800, height=500)

In [33]:
# Linear Regression
y = y.flatten()
lr = LinearRegression()
lr.fit(X, y)

# Support Vector Regression
svr = SVR(kernel='linear')
svr.fit(X, y)

# Evaluate the model
print('LR  Score: ', lr.score(X, y))
print('SVR Score: ', svr.score(X, y))


LR  Score:  0.42600823789139797
SVR Score:  0.4175961161164994


In [35]:
# Polynomial Regression
poly_reg = Pipeline([ ('poly_features', PolynomialFeatures(degree=2, include_bias=False)), ('lin_reg', LinearRegression()), ])
poly_reg.fit(X, y)

# Evaluate the model
print('Polynomial Regression Score: ', poly_reg.score(X, y))

Polynomial Regression Score:  0.8525067519009746


In [36]:
# Polynomial SVR
poly_svr = Pipeline([ ('poly_features', PolynomialFeatures(degree=2, include_bias=False)), ('svr_reg', SVR(kernel='linear'))])
poly_svr.fit(X, y)

# Evaluate the model
print('Polynomial SVR Score: ', poly_svr.score(X, y))

Polynomial SVR Score:  0.8523533814012291


In [42]:
poly_svr = SVR(kernel='poly', degree = 2, C=100000)
poly_svr.fit(X, y)

# Evaluate the model
print('Polynomial Kernel SVR Score: ', poly_svr.score(X, y))

Polynomial Kernel SVR Score:  0.23659795107778703


In [38]:
rbf_svr = SVR(kernel='rbf')
rbf_svr.fit(X, y)

# Evaluate the model
print('Polynomial Kernel SVR Score: ', rbf_svr.score(X, y))

Polynomial Kernel SVR Score:  0.84884525406816


In [43]:
# Visualising the Polynomial Regression and SVR results

x = np.arange(X.min(), X.max(), 0.1).reshape(-1, 1)
y_lr = poly_reg.predict(x)
y_svr = rbf_svr.predict(x)

fig = px.scatter(x=X.flatten(), y=y.flatten(), title='Polynomial Regression', labels={'x': 'X', 'y': 'y'}, width=800, height=500)
fig.add_trace(go.Scatter(x=x.flatten(), y=y_lr.flatten(), mode='lines', line=dict(color='black'), name='Polynomial Regression'))
fig.add_trace(go.Scatter(x=x.flatten(), y=y_svr.flatten(), mode='lines', line=dict(color='yellow'), name='RBF SVR'))

fig.show()
